# 01 — Data Acquisition

Cyclone Biparjoy (Arabian Sea, June 2023): acquire the storm track and satellite rainfall.

**Requirements:** an authenticated Earth Engine session (`earthengine authenticate`) for the IMERG step.
The track step reads a small **illustrative** best-track subset in `data/` (clearly labelled) —
replace it with the official IBTrACS v4 CSV from NOAA NCEI for real work.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd

START, END = '2023-06-07', '2023-06-18'   # Biparjoy window (UTC)
REGION_BBOX = [66.0, 21.0, 70.5, 25.0]    # N Arabian Sea / Sindh–Gujarat landfall region

track = pd.read_csv('../data/biparjoy_besttrack_sample.csv', parse_dates=['time_utc'])
track.head()


## Connect to Earth Engine and load IMERG V07

In [ ]:
import ee
import geemap
from src.gee_utils import initialize_ee, rainfall_sum

# Requires a one-time `earthengine authenticate` in your environment.
initialize_ee()

region = ee.Geometry.Rectangle(REGION_BBOX)
accum = rainfall_sum(START, END, region)
print('Accumulated rainfall image ready:', accum.getInfo()['type'])


## Export accumulated rainfall to GeoTIFF

In [ ]:
out_dir = pathlib.Path('../output'); out_dir.mkdir(exist_ok=True)
geemap.ee_export_image(
    accum,
    filename=str(out_dir / 'biparjoy_imerg_accum_7-18jun2023.tif'),
    scale=10000, region=region, file_per_band=False,
)


**Honesty note:** this notebook documents the real acquisition workflow. Without an EE session it
stops at the EE step — expected. The track-loading cell works offline.
